# Stage 1: Domain Adaptation Training

Notebook ini untuk melatih IndoNanoT5 pada domain Python (Stage 1).

**Prerequisites:** Jalankan `01_setup_and_validation.ipynb` terlebih dahulu.

**Expected Results:**
- Training loss: ~3.0 → ~1.5
- Validation loss: ~2.8 → ~1.8
- Output model: `indot5-python-domain`
- Training time: ~2-3 jam pada T4 GPU

## 1. Setup Environment

In [25]:
# Install dependencies
!pip install -q transformers peft datasets  accelerate
!pip install -q evaluate rouge_score bert_score
print('✓ Dependencies installed')

✓ Dependencies installed


In [26]:
# Cek versi semua library yang terinstall
import importlib
import subprocess
import sys, torch, platform

libs = [
    "transformers",
    "peft", 
    "datasets",
    "accelerate",
    "evaluate",
    "torch",
    "tokenizers",
    "rouge_score",
    "bert_score",
]


print(f"Python:  {sys.version}")
print(f"OS:      {platform.system()}")
print(f"Torch:   {torch.__version__}")
print(f"CUDA:    {torch.cuda.is_available()}")
print(f"\n ")

print("=== Library Versions ===")
for lib in libs:
    try:
        mod = importlib.import_module(lib.replace("-", "_"))
        version = getattr(mod, "__version__", "unknown")
        print(f"  {lib:<20} {version}")
    except ImportError:
        print(f"  {lib:<20} NOT INSTALLED")

# Cek Python version
import sys
print(f"\n  {'python':<20} {sys.version.split()[0]}")

# Cek CUDA
import torch
print(f"  {'cuda available':<20} {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  {'cuda version':<20} {torch.version.cuda}")
    print(f"  {'gpu name':<20} {torch.cuda.get_device_name(0)}")


Python:  3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
OS:      Linux
Torch:   2.10.0+cu128
CUDA:    True

 
=== Library Versions ===
  transformers         5.0.0
  peft                 0.18.1
  datasets             4.0.0
  accelerate           1.13.0
  evaluate             0.4.6
  torch                2.10.0+cu128
  tokenizers           0.22.2
  rouge_score          unknown
  bert_score           0.3.12

  python               3.12.13
  cuda available       True
  cuda version         12.8
  gpu name             Tesla T4


In [27]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('✓ Google Drive mounted')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Google Drive mounted


In [28]:
import torch

# Check GPU
if not torch.cuda.is_available():
    raise RuntimeError('GPU not available! Go to Runtime > Change runtime type > T4 GPU')

gpu_name = torch.cuda.get_device_name(0)
gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'✓ GPU: {gpu_name}')
print(f'✓ Memory: {gpu_memory:.1f} GB')

✓ GPU: Tesla T4
✓ Memory: 15.6 GB


In [30]:
import zipfile, shutil, os

# Hapus folder lama dan extract ulang dari Drive
if os.path.exists('/content/src'):
    shutil.rmtree('/content/src')

shutil.copy('/content/drive/MyDrive/dataset_aqg/src_finetuned.zip', '/content/src_finetuned.zip')
with zipfile.ZipFile('/content/src_finetuned.zip', 'r') as z:
    z.extractall('/content/')
print('✓ src_finetuned.zip extracted ulang')


✓ src_finetuned.zip extracted ulang


## 2. Load Domain Dataset

In [6]:
from src.finetuned.data.dataset_loader import DatasetLoader

loader = DatasetLoader()

# Dataset sudah disalin ke /content/ oleh notebook 01
# Jika sesi baru, salin ulang dari Drive
import os, shutil
DOMAIN_DIR = '/content/dataset_aqg/output_domain/'
if not os.path.exists(DOMAIN_DIR + 'train.jsonl'):
    drive_path = '/content/drive/MyDrive/dataset_aqg/output_domain'
    os.makedirs(DOMAIN_DIR, exist_ok=True)
    for f in ['train.jsonl', 'validation.jsonl', 'test.jsonl']:
        shutil.copy(f'{drive_path}/{f}', f'{DOMAIN_DIR}{f}')
    print('✓ Dataset domain disalin dari Drive')

train_dataset = loader.load_dataset(DOMAIN_DIR, split='train')
val_dataset   = loader.load_dataset(DOMAIN_DIR, split='validation')
test_dataset  = loader.load_dataset(DOMAIN_DIR, split='test')

print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')

✓ Dataset domain disalin dari Drive


Generating train split: 0 examples [00:00, ? examples/s]

✓ Loaded 271 entries from /content/dataset_aqg/output_domain/train.jsonl


Generating train split: 0 examples [00:00, ? examples/s]

✓ Loaded 33 entries from /content/dataset_aqg/output_domain/validation.jsonl


Generating train split: 0 examples [00:00, ? examples/s]

✓ Loaded 36 entries from /content/dataset_aqg/output_domain/test.jsonl
Train: 271 | Val: 33 | Test: 36


In [7]:
# Validate dataset
validation_results = loader.validate_dataset(train_dataset)
print(f"Duplicates: {validation_results['duplicate_count']}")
print(f"Avg input length: {validation_results['avg_input_length']} chars")


=== Dataset Validation Summary ===
Total Entries: 271
Duplicate Count: 18
Avg Input Length: 527.18 chars
Avg Target Length: 286.34 chars
Has Metadata: True
⚠ Warning: Found 18 duplicate entries
Duplicates: 18
Avg input length: 527.18 chars


In [8]:
# Drop kolom metadata sebelum training (fix preprocessing bug)
train_dataset = train_dataset.remove_columns(['metadata'])
val_dataset   = val_dataset.remove_columns(['metadata'])
test_dataset  = test_dataset.remove_columns(['metadata'])
print(f'✓ Kolom: {train_dataset.column_names}')
print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')


✓ Kolom: ['input', 'target']
Train: 271 | Val: 33 | Test: 36


In [9]:
# ========================================
# DEDUPLIKASI DATASET (Recommended)
# ========================================
print('\nDeduplikasi dataset...')
seen_inputs = set()
unique_indices = []
for i, item in enumerate(train_dataset):
    if item['input'] not in seen_inputs:
        seen_inputs.add(item['input'])
        unique_indices.append(i)

train_dataset_dedup = train_dataset.select(unique_indices)
print(f'✓ Deduplikasi: {len(train_dataset)} → {len(train_dataset_dedup)} sampel')
print(f'  Duplikat dihapus: {len(train_dataset) - len(train_dataset_dedup)}')

# Gunakan dataset yang sudah di-deduplikasi
train_dataset = train_dataset_dedup

print(f'\n=== Dataset Final ===')
print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')



Deduplikasi dataset...
✓ Deduplikasi: 271 → 253 sampel
  Duplikat dihapus: 18

=== Dataset Final ===
Train: 253 | Val: 33 | Test: 36


In [10]:
# ========================================
# PRE-TRAINING VERIFICATION
# ========================================
print('\n' + '='*60)
print('PRE-TRAINING VERIFICATION')
print('='*60)

# 1. Dataset size check
print(f'\n1. Dataset Size:')
print(f'   Train: {len(train_dataset)} (expected: 253 after dedup)')
print(f'   Val:   {len(val_dataset)} (expected: 33)')
print(f'   Test:  {len(test_dataset)} (expected: 36)')

# 2. Columns check
print(f'\n2. Columns:')
print(f'   Train: {train_dataset.column_names}')
print(f'   Expected: [\'input\', \'target\']')
if 'metadata' in train_dataset.column_names:
    print('   ❌ ERROR: Kolom metadata masih ada!')
else:
    print('   ✅ OK: Kolom metadata sudah di-drop')

# 3. Sample check
print(f'\n3. Sample Check:')
sample = train_dataset[0]
print(f'   Input length: {len(sample["input"])} chars')
print(f'   Target length: {len(sample["target"])} chars')
print(f'   Input preview: {sample["input"][:100]}...')

print('\n' + '='*60)
print('✅ Verification complete. Ready for training.')
print('='*60 + '\n')



PRE-TRAINING VERIFICATION

1. Dataset Size:
   Train: 253 (expected: 253 after dedup)
   Val:   33 (expected: 33)
   Test:  36 (expected: 36)

2. Columns:
   Train: ['input', 'target']
   Expected: ['input', 'target']
   ✅ OK: Kolom metadata sudah di-drop

3. Sample Check:
   Input length: 36 chars
   Target length: 664 chars
   Input preview: Apa itu Object (Objek) dalam Python?...

✅ Verification complete. Ready for training.



In [11]:
# Preview sample
sample = train_dataset[0]
print('=== Sample Entry ===')
print(f"Input: {sample['input'][:200]}...")
print(f"Target: {sample['target'][:200]}...")
print(f"Format: {sample.get('metadata', {}).get('format', 'N/A')}")

=== Sample Entry ===
Input: Apa itu Object (Objek) dalam Python?...
Target: | Nama | Deskripsi | Contoh |
|------|-----------|--------|
| **Class (Kelas)** | Cetakan (blueprint) untuk membuat objek-objek yang memiliki karakteristik dan perilaku serupa | Mobil; Manusia |
| **O...
Format: N/A


## 3. Setup Model dengan LoRA

In [12]:
from src.finetuned.model.model_setup import ModelSetup
from peft import LoraConfig

MODEL_NAME = 'LazarusNLP/IndoNanoT5-base'

model_setup = ModelSetup()

# LoRA config sesuai spec
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=['q', 'v'],
    bias='none',
    task_type='SEQ_2_SEQ_LM'
)

peft_model, tokenizer = model_setup.setup_model_for_training(
    model_name=MODEL_NAME,
    lora_config=lora_config
)


SETTING UP MODEL FOR TRAINING

Loading base model: LazarusNLP/IndoNanoT5-base


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

...

✓ Model loaded successfully
Loading tokenizer: LazarusNLP/IndoNanoT5-base


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

✓ Tokenizer loaded successfully
Applying LoRA adapters...
  Rank: 8
  Alpha: 16
  Dropout: 0.1
  Target modules: {'q', 'v'}
✓ LoRA adapters applied successfully

=== Model Parameter Summary ===
Trainable params: 884,736 (0.36%)
Total params:     248,462,592
Non-trainable:    247,577,856
✓ Parameter efficiency verified: 0.36% trainable

=== GPU Memory Status ===
Total Memory:     15.64 GB
Allocated Memory: 0.99 GB
Free Memory:      14.64 GB
✓ Sufficient GPU memory available

✓ Model setup complete!


In [13]:
# Analyze token distribution
print('Analyzing token distribution...')
token_stats = loader.analyze_token_distribution(train_dataset, tokenizer, max_length=512)
print(f"Mean length: {token_stats['mean_length']} tokens")
print(f"Exceeding 512: {token_stats['pct_exceeding_limit']}%")

Analyzing token distribution...

=== Token Distribution Analysis ===
Mean Length: 318.31 tokens
Median Length: 113 tokens
Max Length: 2164 tokens
Exceeding 512 tokens: 19.76%

Histogram:
       0-128:  129 █████████████████████████
     128-256:   22 ████
     256-384:   29 █████
     384-512:   23 ████
     512-640:   13 ██
     640-768:   11 ██
    768-1024:    7 █
       1024+:   19 ███

⚠ Warning: 19.76284584980237% of samples exceed max_length=512
Mean length: 318.31 tokens
Exceeding 512: 19.76%


In [14]:
# Pastikan model di GPU
import torch
if torch.cuda.is_available():
    peft_model = peft_model.to('cuda')
    
device = next(peft_model.parameters()).device
print(f'Model device: {device}')
print(f'GPU allocated: {torch.cuda.memory_allocated(0)/1e9:.2f} GB')
print(f'GPU free:      {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0))/1e9:.2f} GB')
print('✓ Model di GPU, siap untuk training')


Model device: cuda:0
GPU allocated: 0.99 GB
GPU free:      14.64 GB
✓ Model di GPU, siap untuk training


In [15]:
# ========================================
# FINAL DATASET VERIFICATION
# ========================================
print('\n' + '='*60)
print('FINAL DATASET VERIFICATION BEFORE TRAINING')
print('='*60)

print(f'\n✓ Dataset ready for training:')
print(f'  Train: {len(train_dataset)} samples')
print(f'  Val:   {len(val_dataset)} samples')
print(f'  Test:  {len(test_dataset)} samples')

print(f'\n✓ Dataset columns (RAW format):')
print(f'  Train columns: {train_dataset.column_names}')
print(f'  Val columns:   {val_dataset.column_names}')

print(f'\n✓ Sample preview:')
sample = train_dataset[0]
print(f'  Input:  {sample["input"][:80]}...')
print(f'  Target: {sample["target"][:80]}...')

print(f'\n⚠️  IMPORTANT: Dataset will be preprocessed by DomainAdaptationTrainer')
print(f'   - Task prefix "question: " will be added automatically')
print(f'   - Tokenization will be handled by trainer.preprocess_dataset()')
print(f'   - Dynamic padding will be applied via DataCollator')

print('\n' + '='*60)
print('✅ Ready to proceed to training configuration')
print('='*60 + '\n')



FINAL DATASET VERIFICATION BEFORE TRAINING

✓ Dataset ready for training:
  Train: 253 samples
  Val:   33 samples
  Test:  36 samples

✓ Dataset columns (RAW format):
  Train columns: ['input', 'target']
  Val columns:   ['input', 'target']

✓ Sample preview:
  Input:  Apa itu Object (Objek) dalam Python?...
  Target: | Nama | Deskripsi | Contoh |
|------|-----------|--------|
| **Class (Kelas)** ...

⚠️  IMPORTANT: Dataset will be preprocessed by DomainAdaptationTrainer
   - Task prefix "question: " will be added automatically
   - Tokenization will be handled by trainer.preprocess_dataset()
   - Dynamic padding will be applied via DataCollator

✅ Ready to proceed to training configuration



## 4. Configure Training

In [16]:
from src.finetuned.training.domain_trainer import DomainAdaptationTrainer

# Checkpoint disimpan ke Drive agar persisten
CHECKPOINT_DIR = '/content/drive/MyDrive/dataset_aqg/checkpoints/domain'

trainer = DomainAdaptationTrainer(
    model=peft_model,
    tokenizer=tokenizer,
    output_dir=CHECKPOINT_DIR,
    max_length=512
)


# Training arguments sesuai spec (Updated for Run 6)
training_args = trainer.get_training_args(
    num_train_epochs=6,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=50,
    logging_steps=10,
    fp16=True
)


print('\n' + '='*60)
print('TRAINING CONFIGURATION')
print('='*60)
print(f'\n✓ Hyperparameters (Spec Defaults):')
print(f'  Epochs:              {training_args.num_train_epochs}')
print(f'  Learning Rate:       {training_args.learning_rate}')
print(f'  Warmup Steps:        {training_args.warmup_steps}')
print(f'  Batch Size:          {training_args.per_device_train_batch_size}')
print(f'  Gradient Accum:      {training_args.gradient_accumulation_steps}')
print(f'  Effective Batch:     {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}')
print(f'  FP16:                {training_args.fp16}')

print(f'\n✓ Training Estimates:')
print(f'  Steps per epoch:     ~{len(train_dataset) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)}')
print(f'  Total steps:         ~{(len(train_dataset) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)) * training_args.num_train_epochs}')
print(f'  Warmup duration:     ~{training_args.warmup_steps / (len(train_dataset) // 16):.1f} epochs')
print(f'  Estimated time:      30-45 minutes on T4 GPU')

print(f'\n✓ Checkpoint Directory:')
print(f'  {CHECKPOINT_DIR}')

print('\n' + '='*60)
print('✅ Configuration ready for training')
print('='*60 + '\n')




TRAINING CONFIGURATION

✓ Hyperparameters (Spec Defaults):
  Epochs:              6
  Learning Rate:       0.0002
  Warmup Steps:        50
  Batch Size:          4
  Gradient Accum:      4
  Effective Batch:     16
  FP16:                True

✓ Training Estimates:
  Steps per epoch:     ~15
  Total steps:         ~90
  Warmup duration:     ~3.3 epochs
  Estimated time:      30-45 minutes on T4 GPU

✓ Checkpoint Directory:
  /content/drive/MyDrive/dataset_aqg/checkpoints/domain

✅ Configuration ready for training



## 5. Start Training

> ⚠️ Training akan memakan waktu ~2-3 jam. Pastikan Colab tidak timeout.
> Checkpoints disimpan setiap epoch ke Google Drive.

In [17]:
import time
start_time = time.time()

print('Starting domain adaptation training...')
print('Checkpoints will be saved to:', CHECKPOINT_DIR)
print('='*60)

results = trainer.train(
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    training_args=training_args,
    early_stopping=True,
    early_stopping_patience=2
)

elapsed = (time.time() - start_time) / 3600
print(f'\nTraining completed in {elapsed:.2f} hours')
print(f'Final training loss: {results["training_loss"]:.4f}')

Starting domain adaptation training...
Checkpoints will be saved to: /content/drive/MyDrive/dataset_aqg/checkpoints/domain

STARTING DOMAIN ADAPTATION TRAINING

✓ Model moved to GPU: Tesla T4
  Model device: cuda:0
Preprocessing datasets...
Preprocessing 253 samples...
  Columns: ['input', 'target']
  Batch size: 32
  Removing columns: ['input', 'target']


Tokenizing:   0%|          | 0/253 [00:00<?, ? examples/s]

✓ Preprocessed 253 samples
  Sample label check: 217 valid tokens, 0 masked (-100)
Preprocessing 33 samples...
  Columns: ['input', 'target']
  Batch size: 32
  Removing columns: ['input', 'target']


Tokenizing:   0%|          | 0/33 [00:00<?, ? examples/s]

✓ Preprocessed 33 samples
  Sample label check: 24 valid tokens, 0 masked (-100)

=== Dataset Size After Preprocessing ===
Train samples (actual): 253
Eval samples (actual):  33

=== Training Configuration ===
Epochs: 6
Batch size: 4
Gradient accumulation: 4
Effective batch size: 16
Learning rate: 0.0002
Warmup steps: 50
FP16: True
Train samples: 253
Eval samples: 33
Starting training...


Epoch,Training Loss,Validation Loss
1,40.300360,10.033830
2,40.125375,9.933958
3,39.789474,9.690681
4,38.591214,9.467672
5,37.675067,9.331713
6,37.381027,9.285494



=== Training Complete ===
Final training loss: 38.9387
Training time: 203.85 seconds
Training samples per second: 7.45
✓ Training results saved to /content/drive/MyDrive/dataset_aqg/checkpoints/domain/training_results.json

Training completed in 0.06 hours
Final training loss: 38.9387


## 6. Save Best Model

In [18]:
# Save best model
model_path = trainer.save_best_model(output_name='indot5-python-domain')
print(f'Model saved to: {model_path}')

# Plot training curves
trainer.plot_training_curves(
    save_path=f'{CHECKPOINT_DIR}/training_curves.png'
)
print('✓ Training curves saved')


✓ Best model saved to: /content/drive/MyDrive/dataset_aqg/checkpoints/domain/indot5-python-domain
Model saved to: /content/drive/MyDrive/dataset_aqg/checkpoints/domain/indot5-python-domain
✓ Training curves saved to /content/drive/MyDrive/dataset_aqg/checkpoints/domain/training_curves.png
✓ Training curves saved


## 7. Evaluate on Validation Set

In [19]:
from src.finetuned.evaluation.metrics_calculator import MetricsCalculator
from src.finetuned.evaluation.model_evaluator import ModelEvaluator

metrics_calc = MetricsCalculator()
evaluator = ModelEvaluator(
    model=peft_model,
    tokenizer=tokenizer,
    metrics_calculator=metrics_calc
)
print('✓ Evaluator ready')

✓ Evaluator ready


In [20]:
# ========================================
# INFERENCE TEST WITH TASK PREFIX
# ========================================
print('\n' + '='*60)
print('INFERENCE TEST')
print('='*60)

test_inputs = [
    "Apa itu list dalam Python?",
    "Jelaskan fungsi dalam Python.",
    "Apa itu variable dalam Python?"
]

# FIX: Tidak menggunakan prefix "question:" untuk domain adaptation.
# Konsisten dengan preprocessing training — input di-lowercase tanpa prefix tambahan.
# Prefix "question:" dihapus karena domain dataset berisi dua format (qa_generic &
# span_corruption) yang tidak memerlukan task prefix eksplisit.

for inp in test_inputs:
    # Lowercase saja, tanpa prefix — konsisten dengan training
    processed_input = inp.lower()
    
    inputs = tokenizer(
        processed_input, 
        return_tensors="pt", 
        max_length=512, 
        truncation=True
    ).to("cuda")
    
    # FIX: Tambahkan repetition_penalty dan no_repeat_ngram_size
    # untuk mencegah output berulang (dash, kata, atau frasa yang diulang terus).
    # repetition_penalty=1.5 : token yang sudah muncul di-penalize saat sampling
    # no_repeat_ngram_size=3 : larang 3-gram yang sama muncul dua kali
    outputs = peft_model.generate(
        **inputs, 
        max_new_tokens=256,
        num_beams=4,
        no_repeat_ngram_size=3,
        repetition_penalty=1.5,
        early_stopping=True
    )
    
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    print(f'Input:  {inp}')
    print(f'Output: {result}')
    print('-' * 60)

print('\n✅ Inference test complete')
print('='*60 + '\n')



INFERENCE TEST

⚠️  IMPORTANT: Adding task prefix "question: " for inference
   (T5 models require task prefix for correct output)

Input:  Apa itu list dalam Python?
Output: - – - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
------------------------------------------------------------
Input:  Jelaskan fungsi dalam Python.
Output: - - – - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
------------------------------------------------------------
Input:  Apa itu variable dalam Python?
Output: - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -

In [21]:
import os
from peft import PeftModel
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

MODEL_PATH = f'{CHECKPOINT_DIR}/indot5-python-domain'

if os.path.exists(MODEL_PATH):
    print(f'Loading model from checkpoint: {MODEL_PATH}')
    base_model = AutoModelForSeq2SeqLM.from_pretrained('LazarusNLP/IndoNanoT5-base')
    peft_model = PeftModel.from_pretrained(base_model, MODEL_PATH)
    if torch.cuda.is_available():
        peft_model = peft_model.cuda()
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    print(f'✓ Model loaded | device: {next(peft_model.parameters()).device}')

    # Re-init evaluator dengan model yang baru di-load
    metrics_calc = MetricsCalculator()
    evaluator = ModelEvaluator(
        model=peft_model,
        tokenizer=tokenizer,
        metrics_calculator=metrics_calc
    )
    print('✓ Evaluator re-initialized')
else:
    print(f'⚠ Checkpoint tidak ditemukan di: {MODEL_PATH}')
    print('Pastikan training sudah selesai dan model sudah di-save.')

Loading model from checkpoint: /content/drive/MyDrive/dataset_aqg/checkpoints/domain/indot5-python-domain


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✓ Model loaded | device: cuda:0
✓ Evaluator re-initialized


In [22]:
print('Evaluating on validation set...')
val_metrics = evaluator.evaluate_on_test_set(
    test_dataset=val_dataset,
    num_beams=4,
    include_bertscore=False,
    max_samples=50
)

print('\n=== Validation Metrics ===')
print(f'  BLEU-4:  {val_metrics.get("bleu_4", 0):.4f}')
print(f'  ROUGE-L: {val_metrics.get("rouge_l", 0):.4f}')
print(f'  ROUGE-1: {val_metrics.get("rouge_1", 0):.4f}')

The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Evaluating on validation set...

EVALUATING ON TEST SET

Evaluating 33 samples...
  Processed 10/33 samples...
  Processed 20/33 samples...
  Processed 30/33 samples...
✓ Generated 33 predictions
Computing metrics for 33 samples...
  Computing BLEU...


  Computing ROUGE...


  Computing Diversity...
✓ All metrics computed

Test Set Evaluation Results

BLEU Scores:
  BLEU:     0.0000
  BLEU-1:   0.0700
  BLEU-2:   0.0140
  BLEU-3:   0.0003
  BLEU-4:   0.0000

ROUGE Scores:
  ROUGE-1:  0.1203
  ROUGE-2:  0.0018
  ROUGE-L:  0.0882

Diversity:
  Distinct-1: 0.1253
  Distinct-2: 0.6875


=== Validation Metrics ===
  BLEU-4:  0.0000
  ROUGE-L: 0.0882
  ROUGE-1: 0.1203


In [23]:
print('Evaluating on test set (final evaluation)...')
test_metrics = evaluator.evaluate_on_test_set(
    test_dataset=test_dataset,
    num_beams=4,
    include_bertscore=False,
    max_samples=None  # evaluasi semua 36 sampel
)

print('\n=== Test Set Metrics (Final) ===')
print(f'  BLEU-4:  {test_metrics.get("bleu_4", 0):.4f}')
print(f'  ROUGE-L: {test_metrics.get("rouge_l", 0):.4f}')
print(f'  ROUGE-1: {test_metrics.get("rouge_1", 0):.4f}')

Evaluating on test set (final evaluation)...

EVALUATING ON TEST SET

Evaluating 36 samples...
  Processed 10/36 samples...
  Processed 20/36 samples...
  Processed 30/36 samples...
✓ Generated 36 predictions
Computing metrics for 36 samples...
  Computing BLEU...
  Computing ROUGE...
  Computing Diversity...
✓ All metrics computed

Test Set Evaluation Results

BLEU Scores:
  BLEU:     0.0000
  BLEU-1:   0.0815
  BLEU-2:   0.0165
  BLEU-3:   0.0012
  BLEU-4:   0.0000

ROUGE Scores:
  ROUGE-1:  0.1163
  ROUGE-2:  0.0003
  ROUGE-L:  0.0884

Diversity:
  Distinct-1: 0.1152
  Distinct-2: 0.6516


=== Test Set Metrics (Final) ===
  BLEU-4:  0.0000
  ROUGE-L: 0.0884
  ROUGE-1: 0.1163


In [24]:
print('\n' + '='*60)
print('DOMAIN ADAPTATION TRAINING SUMMARY')
print('='*60)
print(f'Final Training Loss: {results["training_loss"]:.4f}')
print(f'Training Time:       {elapsed:.2f} hours')
print(f'Model saved:         {model_path}')
print(f'\nValidation Metrics (monitor):')
print(f'  BLEU-4:  {val_metrics.get("bleu_4", 0):.4f}')
print(f'  ROUGE-L: {val_metrics.get("rouge_l", 0):.4f}')
print(f'\nTest Set Metrics (final):')
print(f'  BLEU-4:  {test_metrics.get("bleu_4", 0):.4f}')
print(f'  ROUGE-L: {test_metrics.get("rouge_l", 0):.4f}')
print('\n✓ Stage 1 complete! Proceed to 03_task_specific_training.ipynb')


DOMAIN ADAPTATION TRAINING SUMMARY
Final Training Loss: 38.9387
Training Time:       0.06 hours
Model saved:         /content/drive/MyDrive/dataset_aqg/checkpoints/domain/indot5-python-domain

Validation Metrics (monitor):
  BLEU-4:  0.0000
  ROUGE-L: 0.0882

Test Set Metrics (final):
  BLEU-4:  0.0000
  ROUGE-L: 0.0884

✓ Stage 1 complete! Proceed to 03_task_specific_training.ipynb
